# Getting Query Tensor
---

This notebook demonstrates an unrefined user-facing API to get Query tensor for
testing store Q implementations. This notebook and example, overall, will be
refined in future PR. Particularly, this notebook still uses a separate context
for QCache, which is a thin wrapper over LMCacheKVCacheContext.

Requirements:
* This experiment requires a GPU. To demonstrate how token dropping increases 
  the decoding batch size, adjust the GPU memory utilization together with the 
  number of requests. This example uses one RTX 6000 PRO.
* This example uses shared memory for data transfer between LMCache server and
  the SDK. If shared memory is unavailable, the SDK automatically falls back to pickle.

## Setup vLLM and LMCache server
Follow below instructions (1-4), then proceed to below Python cell.

**1. Start LMCache server first**

To use shared memory, specify the `--shm-name` and `--no-l1-use-lazy`

```sh
lmcache server \
    --l1-size-gb 150 \
    --eviction-policy LRU \
    --chunk-size 256 \
    --port 6555 \
    --http-port 8080 \
    --shm-name lmcache_kvcache_sdk_e2e \
    --no-l1-use-lazy
```

**2. Wait until LMCache server is ready**

```sh
curl -sf http://localhost:8080/healthcheck && echo " LMCache ready"
```

**3. Start vLLM once LMCache server is ready**

We pass --no-enable-prefix-caching to disable vLLM's built-in prefix caching. 
This ensures the prefilled KV cache is always served from LMCache rather than 
vLLM, so the decoding throughput improvement can be attributed entirely to the 
tokens dropped through LMCache.

```sh
vllm serve Qwen/Qwen3-8B \
    --port 8000 \
    --served-model-name Qwen/Qwen3-8B \
    --no-enable-prefix-caching \
    --enforce-eager \
    --gpu-memory-utilization 0.65 \
    --kv-transfer-config '{"kv_connector":"LMCacheMPConnector","kv_role":"kv_both","kv_connector_extra_config":{"lmcache.mp.port":6555}}' \
    --trust-remote-code \
    --return-tokens-as-token-ids
```

**4. Wait until vLLM is ready**

```sh
curl -sf http://localhost:8000/v1/models && echo " vLLM ready"
```

## Run E2E KV Edit Example

In [2]:
# SPDX-License-Identifier: Apache-2.0
"""End-to-end KV cache remapping driver for the SDK example."""

# Standard
from datasets import load_dataset
import sys
from transformers import AutoTokenizer, AutoConfig

# Third Party
import torch

# First Party
from lmcache.logging import init_logger
import lmcache.sdk.kvcache as lmc_sdk
import lmcache.sdk.stream as lmc_stream
import lmcache.sdk.batch as lmc_batch
from lmcache.banner import print_banner_once
from utils import rerotate_k_cache, make_post_completion

print_banner_once(sys.stdout)
logger = init_logger(__name__)

/home/rani/LMCache/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-07-07 22:38:11,875] LMCache INFO: torch_dev=<module 'torch.cuda' from '/home/rani/LMCache/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py'>, torch_device_type=cuda (__init__.py:208:lmcache.v1.platform)
[2026-07-07 22:38:12,183] LMCache INFO: Using backend: lmcache.c_ops (__init__.py:198:lmcache.v1.platform)
[2026-07-07 22:38:12,252] LMCache INFO: multi_layer_block_kv_transfer mode: ptr (base.py:94:lmcache.v1.multiprocess.transfer_context.base)



 _     __  __    ____           _          
| |   |  \/  |  / ___|__ _  ___| |__   ___      LMCache v0.5.1rc3.dev6 (g30352f22)
| |   | |\/| | | |   / _` |/ __| '_ \ / _ \     Website:  https://lmcache.ai/
| |___| |  | | | |__| (_| | (__| | | |  __/     Recipes:  https://docs.lmcache.ai/recipes
|_____|_|  |_|  \____\__,_|\___|_| |_|\___|     LinkedIn: https://www.linkedin.com/company/lmcache-lab
Set LMCACHE_DISABLE_BANNER=1 to hide this banner.



## Setting up hyperparameters

In [3]:
model_name = "Qwen/Qwen3-8B"
vllm_url = "http://localhost:8000"
lmcache_url = "http://localhost:8081"
lmcache_mq_url = "tcp://localhost:6555"
chunk_size = 256
max_tokens = 5120
timeout = 60  # timeout for context retrieval (seconds)
trust_remote_code = True

## Configuring the model and LMCache endpoint

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name, trust_remote_code=trust_remote_code
)
config = AutoConfig.from_pretrained(model_name, trust_remote_code=trust_remote_code)
head_size = getattr(
    config, "head_dim", config.hidden_size // config.num_attention_heads
)
work_device = torch.device("cpu")
post_completion = make_post_completion(vllm_url, model_name, timeout)
ctx = lmc_sdk.connect(
    url=lmcache_mq_url,
    http_url=lmcache_url,
    model_name=model_name,
    timeout=timeout,
)

[2026-07-07 22:38:16,832] LMCache INFO: Initialized LMCacheKVCacheContext with instance_id=1593071, model_name=Qwen/Qwen3-8B, chunk_size=256, shm_name=lmcache_l1_pool_lmcache_kvcache_sdk_e2e (kvcache.py:114:lmcache.sdk.kvcache)
[2026-07-07 22:38:16,835] LMCache INFO: Creating transfer context (device_type=cpu, mode=auto) (worker_transfer.py:641:lmcache.v1.multiprocess.transfer_context.worker_transfer)
[2026-07-07 22:38:16,927] LMCache INFO: Using AsyncEngineDrivenTransferContext for store path (worker_transfer.py:101:lmcache.v1.multiprocess.transfer_context.worker_transfer)
[2026-07-07 22:38:16,928] LMCache INFO: Engine KV Format: EngineKVFormat.NL_X_NB_TWO_NH_BS_HS NL x [NB, 2, NH, BS, HS] (detection.py:44:lmcache.v1.gpu_connector.kv_format.detection)
[2026-07-07 22:38:16,929] LMCache INFO: Creating EngineDrivenContextShm (shm_name=lmcache_l1_pool_lmcache_kvcache_sdk_e2e, pool_size=161061273600) (base.py:235:lmcache.v1.multiprocess.transfer_context.base)
[2026-07-07 22:38:16,992] LMCa

## Constructing Prompt

In [5]:
prompts = []
ds = load_dataset("raniayu/token-dropping-demo", split="train")
for i, example in enumerate(ds):
    if i >= 10:
        break
    prompt = tokenizer.encode(ds[i]["prompt"], return_tensors="pt").squeeze(0).tolist()
    prompts.append(prompt)

print(f"Loaded {len(prompts)} prompts from the dataset.")
print(f"Total prompts length: {sum(len(p) for p in prompts)} tokens.")
print(f"Mean prompt length: {sum(len(p) for p in prompts) / len(prompts):.2f} tokens.")
print(f"Max prompt length: {max(len(p) for p in prompts)} tokens.")
print(f"Min prompt length: {min(len(p) for p in prompts)} tokens.")

Loaded 10 prompts from the dataset.
Total prompts length: 112941 tokens.
Mean prompt length: 11294.10 tokens.
Max prompt length: 12903 tokens.
Min prompt length: 10007 tokens.


## With Token Dropping

In [6]:
def drop_tokens_fn(
    kv_tensor: torch.Tensor, token_source: list[int]
) -> tuple[torch.Tensor, list[int]]:
    """Drop the middle half of the chunks, keeping the first and last intact.

    Args:
        kv_tensor: KV cache with shape [2, L, T, D]
        token_source: Token ids
    Returns:
        A tuple of the compacted KV tensor (on CPU) and the token ids.
    Raises:
        ValueError: If there are fewer than 3 chunks.
    """
    h = kv_tensor.shape[2]

    num_chunks = (h + chunk_size - 1) // chunk_size
    drop_count = num_chunks // 2
    if num_chunks < 3:
        raise ValueError("Not enough chunks to drop.")

    drop_start = (num_chunks - drop_count) // 2
    drop_start = max(1, min(drop_start, num_chunks - 1 - drop_count))

    lo = drop_start * chunk_size
    hi = (drop_start + drop_count) * chunk_size
    keep_idx = torch.cat([torch.arange(lo), torch.arange(hi, h)])
    kept_ids = token_source[:lo] + token_source[hi:]
    logger.info(f"compacting {drop_count}/{num_chunks} chunks")

    e_kv = rerotate_k_cache(
        kv_tensor[:, :, keep_idx, :].clone().to(work_device),
        old_positions=keep_idx.to(work_device),
        new_positions=torch.arange(
            keep_idx.numel(), device=work_device, dtype=torch.long
        ),
        model_config=config,
    )

    assert keep_idx[0].item() == 0 and keep_idx[-1].item() == h - 1
    return e_kv.cpu(), kept_ids

In [ ]:
batch = lmc_batch.LMCacheBatchedStream()
for i, prompt in enumerate(prompts):
    stream = lmc_stream.create_request(
        ctx=ctx,
        post_completion=post_completion,
        prompt_token_ids=prompt,
    )
    batch.add(stream)

results = batch.prefill(
    sampling_params={"max_tokens": 1, "temperature": 1.0, "ignore_eos": True}
)
results.emit()

======================= Batched Stream Metrics (prefill) =======================
-------------------------------- Configuration ---------------------------------
Number of Streams:                                                            10
----------------------------------- Results ------------------------------------
Total Duration (s):                                                         0.74
Total Input Tokens:                                                       112941
Input Throughput (tokens/s):                                           153495.74





## Getting Query Tensor

In [ ]:
import lmcache.sdk.qcache as lmc_qcache_sdk

qctx = lmc_qcache_sdk.connect(
    url=lmcache_mq_url,
    http_url=lmcache_url,
    model_name=model_name,
    timeout=timeout,
)

print(len(prompts[0]))
print(lmc_qcache_sdk.retrieve_query(qctx, prompts[0]))

[2026-07-07 22:38:58,353] LMCache INFO: Initialized LMCacheKVCacheContext with instance_id=1593071, model_name=__lmc_query__Qwen/Qwen3-8B, chunk_size=256, shm_name=lmcache_l1_pool_lmcache_kvcache_sdk_e2e (kvcache.py:114:lmcache.sdk.kvcache)
[2026-07-07 22:38:58,394] LMCache INFO: Creating transfer context (device_type=cpu, mode=auto) (worker_transfer.py:641:lmcache.v1.multiprocess.transfer_context.worker_transfer)
[2026-07-07 22:38:58,395] LMCache INFO: Using AsyncEngineDrivenTransferContext for store path (worker_transfer.py:101:lmcache.v1.multiprocess.transfer_context.worker_transfer)
[2026-07-07 22:38:58,396] LMCache INFO: Engine KV Format: EngineKVFormat.NL_X_NB_TWO_NH_BS_HS NL x [NB, 2, NH, BS, HS] (detection.py:44:lmcache.v1.gpu_connector.kv_format.detection)
[2026-07-07 22:38:58,398] LMCache INFO: Creating EngineDrivenContextShm (shm_name=lmcache_l1_pool_lmcache_kvcache_sdk_e2e, pool_size=161061273600) (base.py:235:lmcache.v1.multiprocess.transfer_context.base)
[2026-07-07 22:39

10136
tensor([[[[ 4.2188e-01, -5.4688e-01, -3.6133e-01,  ...,  1.0547e+00,
           -1.3477e-01,  7.2266e-01],
          [ 2.0605e-01, -1.3477e-01,  3.5938e-01,  ..., -4.6680e-01,
            9.6094e-01, -2.3594e+00],
          [-1.4526e-02, -1.1816e-01,  4.1406e-01,  ..., -1.4453e+00,
            5.9082e-02, -1.1953e+00],
          ...,
          [ 8.6060e-03, -1.1719e-01, -1.7188e-01,  ...,  3.3203e-01,
           -1.5820e-01,  4.6680e-01],
          [ 1.4160e-01, -2.3242e-01, -2.1191e-01,  ..., -1.5000e+00,
            5.5078e-01, -1.0859e+00],
          [-1.5564e-02, -9.4238e-02,  6.5918e-03,  ..., -8.8281e-01,
            4.1211e-01,  1.1953e+00]],

         [[ 1.1084e-01, -1.2188e+00, -1.6641e+00,  ...,  2.3906e+00,
           -1.8672e+00,  1.7422e+00],
          [ 2.3633e-01,  1.5547e+00, -2.8906e+00,  ..., -2.7734e-01,
            3.9648e-01, -1.1768e-01],
          [ 1.0469e+00, -1.5156e+00, -2.0469e+00,  ...,  4.6875e-01,
           -3.8672e-01,  5.7422e-01],
          ...,

## Close both contexts

In [10]:
# Only close the context when done.
lmc_sdk.close(ctx)
lmc_sdk.close(qctx)